In [1]:
import pandas as pd
import sys
import numpy as np

In [2]:
sys.path.insert(0, '')
from Clase_Valor import Valor
from Transversal import leer_activos, leer_parametros, leer_configuracion

In [3]:
carpeta_input, cofre, seguimiento, saving_step = leer_parametros()
output_level = leer_configuracion()
output_level

In [4]:
eliminar_varianzas_directas = False

In [5]:
df_activos = leer_activos(carpeta_input)

Small batch 240812


In [6]:
cofre0 = '/'.join(cofre.split('/')[:-2]) + '/'
for i in range(len(df_activos)):
    simbolo, nombre = df_activos.loc[i]
    valor = Valor(simbolo, nombre, cofre) # Si no existe el objeto, se crea 
    #display(valor.adj_data.tail())
    if i == 0:
        df_rendimientos = valor.adj_data[['Date', 'rendimiento']].rename(columns = {'rendimiento': simbolo})
    else:
        df_rendimientos = df_rendimientos.merge(valor.adj_data[['Date', 'rendimiento']].rename(columns = {'rendimiento': simbolo}), on = 'Date', how = 'inner')
df_rendimientos = df_rendimientos.fillna(0)
df_rendimientos

NFLX.pkl ../Cofre/Valor/
../Cofre/Valor/NFLX.pkl
FCEL.pkl ../Cofre/Valor/
../Cofre/Valor/FCEL.pkl
DVA.pkl ../Cofre/Valor/
../Cofre/Valor/DVA.pkl
AAPL.pkl ../Cofre/Valor/
../Cofre/Valor/AAPL.pkl
CAT.pkl ../Cofre/Valor/
../Cofre/Valor/CAT.pkl


,Date,NFLX,FCEL,DVA,AAPL,CAT
0,2002-05-23,0.000000,-0.050681,0.000457,0.035361,-0.000369
1,2002-05-24,0.011343,-0.010359,0.017833,-0.040906,-0.005530
2,2002-05-25,-0.010921,0.009259,0.004380,-0.001760,-0.002132
3,2002-05-26,-0.011041,0.009174,0.004361,-0.001763,-0.002136
4,2002-05-27,-0.011165,0.009091,0.004342,-0.001766,-0.002141
...,...,...,...,...,...,...
8129,2024-08-24,0.000830,-0.007407,0.001645,0.000500,0.002631
8130,2024-08-25,0.000829,-0.007463,0.001642,0.000499,0.002624
8131,2024-08-26,0.000829,-0.007519,0.001640,0.000499,0.002617
8132,2024-08-27,0.010575,-0.045455,0.000646,0.003742,-0.000114


In [7]:
def obtener_covarianza(df_rendimientos_n, campos_output, df_activos, eliminar_varianzas_directas, n):
    
    all_dict = {}
    if len(df_rendimientos_n) != n:
        for c in campos_output:
            all_dict[c] = np.nan
        df_output = pd.Series(all_dict)
        return df_output
    
    df_rendimientos_n = df_rendimientos_n.drop(columns = 'Date')

    #display(df_activos)
    
    df_activos_id = df_activos[['SIMBOLO']].copy()
    df_activos_id['ID'] = range(len(df_activos_id))
    # obtener matriz de covarianza
    df_cov = df_rendimientos_n.cov()

    #pivotear y que los nombres de los campos finales sean f'{i}-{j}'
    df_cov = df_cov.stack().reset_index()
    df_cov = df_cov.merge(df_activos_id, left_on = 'level_0', right_on = 'SIMBOLO', how = 'left').drop(columns = ['SIMBOLO'])
    df_cov = df_cov.rename(columns = {'ID': 'level_0_id'})
    df_cov = df_cov.merge(df_activos_id, left_on = 'level_1', right_on = 'SIMBOLO', how = 'left').drop(columns = ['SIMBOLO'])
    df_cov = df_cov.rename(columns = {'ID': 'level_1_id'})
    
    #display(df_cov)

    if eliminar_varianzas_directas:
        df_cov = df_cov[df_cov['level_0_id'] > df_cov['level_1_id']].reset_index(drop = True) # Se eliminan las varianzas directas (solo quedan los casos con i distinto de j)
    else:
        df_cov = df_cov[df_cov['level_0_id'] >= df_cov['level_1_id']].reset_index(drop = True) # Se eliminan las varianzas directas (solo quedan los casos con i distinto de j)
    
    df_cov = df_cov.drop(columns = ['level_0_id', 'level_1_id'])
    
    #display(df_cov)
    
    df_cov['index'] = df_cov['level_0'] + '-' + df_cov['level_1']
    df_cov = df_cov.drop(columns = ['level_0', 'level_1'])
    df_cov = df_cov.set_index('index').T
    
    #sys.exit()

    for i in df_cov.columns:
        all_dict[i] = df_cov[i].values[0]
    
    if len(all_dict) == 0:
        for c in campos_output:
            all_dict[c] = np.nan
    
    #print(all_dict)
    
    df_output = pd.Series(all_dict)
    #display(df_output)
    return df_output

def sub_df(df, date, n, modo, campos_output, df_activos, eliminar_varianzas_directas):
    if modo == 'output':
        df = df[df['Date'] > date].head(n).reset_index(drop = True)
    else:
        df = df[df['Date'] <= date].tail(n).reset_index(drop = True)
    
    df_cov = obtener_covarianza(df, campos_output, df_activos, eliminar_varianzas_directas, n)
    
    return df_cov

In [8]:
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import cvxpy as cp
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

def RLM(df_dates, dias_inputs, output = True):
    # Resuelve RLM como una ponderacion: Suma(coef) = 1 y coef >= 0
    # Supongo que ya tienes df_dates, dias_inputs, etc. definidos.
    df_dates_train, df_dates_test = train_test_split(df_dates, test_size=0.2)

    x_train = df_dates_train[[f'input{k}' for k in dias_inputs]].values
    y_train = df_dates_train['output'].values
    x_test = df_dates_test[[f'input{k}' for k in dias_inputs]].values
    y_test = df_dates_test['output'].values

    # Variables de optimización
    n_features = x_train.shape[1]
    beta = cp.Variable(n_features)

    # Función de pérdida
    y_pred = x_train @ beta
    loss = cp.sum_squares(y_train - y_pred)

    # Restricción: La suma de los coeficientes debe ser igual a 1
    constraints = [cp.sum(beta) == 1]

    # Restriccion 2: Todos los coeficientes deben ser positivos (+ restriccion de arriba = ponderación)
    constraints += [beta >= 0]

    # Problema de optimización
    problem = cp.Problem(cp.Minimize(loss), constraints)

    # Resolver el problema
    problem.solve()

    # Obtener los valores de los coeficientes
    beta_values = beta.value

    # Imprimir los coeficientes
    if output:
        print("Coeficientes:", beta_values)

    # Predecir con los datos de prueba
    y_test_pred = x_test @ beta_values

    # Evaluar el rendimiento del modelo
    mse = mean_squared_error(y_test, y_test_pred)
    if output:
        print("Error cuadrático medio:", mse)
    return beta_values, mse

In [9]:
lista_activos = df_rendimientos.columns.tolist()
lista_activos.remove('Date')
L = len(lista_activos)

if eliminar_varianzas_directas:
    campos_output = [f'{lista_activos[i]}-{lista_activos[j]}' for i in range(L) for j in range(L) if i > j]
else:
    campos_output = [f'{lista_activos[i]}-{lista_activos[j]}' for i in range(L) for j in range(L) if i >= j]
campos_output

['NFLX-NFLX',
 'FCEL-NFLX',
 'FCEL-FCEL',
 'DVA-NFLX',
 'DVA-FCEL',
 'DVA-DVA',
 'AAPL-NFLX',
 'AAPL-FCEL',
 'AAPL-DVA',
 'AAPL-AAPL',
 'CAT-NFLX',
 'CAT-FCEL',
 'CAT-DVA',
 'CAT-AAPL',
 'CAT-CAT']

In [10]:
def correccion(df_dates, df_rendimientos, campos_output, nombre):
    df_output = df_rendimientos[['Date'] + campos_output].copy()
    for c in campos_output:
        df_output = df_output.rename(columns = {c: f'{nombre}_{c}'})
        df_rendimientos = df_rendimientos.drop(columns = c)
    df_dates = df_dates.merge(df_output, on = 'Date', how = 'left')
    return df_dates, df_rendimientos

In [11]:
#df_rendimientos = df_rendimientos[['Date', 'NFLX', 'AAPL', 'DOW']] # small batch
df_rendimientos = df_rendimientos.reset_index(drop = True)#.head(20) # small batch

df_dates = df_rendimientos[['Date']]
#sys.exit()
# continuar aquí 240618 (anexar campos output)
df_rendimientos[campos_output] = df_rendimientos['Date'].apply(lambda x: sub_df(df_rendimientos, x, output_level, 'output', campos_output, df_activos, eliminar_varianzas_directas))
df_dates, df_rendimientos = correccion(df_dates, df_rendimientos, campos_output, 'output')

dias_inputs = [30, 45, 60, 75, 90]
#dias_inputs = [3, 5, 10]
for k in dias_inputs:
    df_rendimientos[campos_output] = df_rendimientos['Date'].apply(lambda x: sub_df(df_rendimientos, x, k, 'input', campos_output, df_activos, eliminar_varianzas_directas))
    df_dates, df_rendimientos = correccion(df_dates, df_rendimientos, campos_output, f'input{k}')
    #display(df_dates)

In [12]:
df_dates = df_dates.melt(id_vars = 'Date', var_name = 'ID', value_name = 'COV')
df_dates[['NAME', 'VALORES']] = df_dates['ID'].str.split('_', expand = True)
df_dates = df_dates.drop(columns = 'ID')
df_dates = df_dates.pivot(index = ['Date', 'VALORES'], columns = 'NAME', values = 'COV').reset_index()

for k in dias_inputs:
    df_dates = df_dates[df_dates[f'input{k}'].notna()]#.reset_index(drop = True)

df_dates_predict = df_dates[df_dates['output'].isna()].reset_index(drop = True)
df_dates = df_dates[(df_dates['output'].notna())].reset_index(drop = True)
#df_dates = df_dates.drop(columns = ['input_min', 'input_max'])
df_dates = df_dates[['Date'] + [f'input{k}' for k in dias_inputs] + ['output']]
df_dates_predict = df_dates_predict[['Date', 'VALORES'] + [f'input{k}' for k in dias_inputs] + ['output']]
df_dates.head()

NAME,Date,input30,input45,input60,input75,input90,output
0,2002-08-20,0.000589,0.000940,0.000845,0.001046,0.000922,0.000217
1,2002-08-20,0.000133,0.000106,0.000076,-0.000022,-0.000024,0.000084
2,2002-08-20,0.000573,0.000295,0.000381,0.000425,0.000343,0.000280
3,2002-08-20,0.000511,0.000532,0.000443,0.000255,0.000190,0.000181
4,2002-08-20,0.000384,0.000269,0.000242,0.000237,0.000200,0.000021


In [13]:
# Continuar aqui 240621: Listo para ejecutar RLM
beta_values, mse = RLM(df_dates, dias_inputs)

Coeficientes: [-1.65570105e-21  3.60186205e-02  9.45411087e-02  2.00064965e-01
  6.69375305e-01]
Error cuadrático medio: 2.436580550787125e-06


In [14]:
df_dates_predict = df_dates_predict[df_dates_predict['Date'] == df_dates_predict['Date'].max()]
df_dates_predict['output_predict'] = df_dates_predict[[f'input{k}' for k in dias_inputs]].values @ beta_values
df_dates_predict

NAME,Date,VALORES,input30,input45,input60,input75,input90,output,output_predict
435,2024-08-28,AAPL-AAPL,0.000096,0.000109,0.000112,1.082741e-04,0.000161,NaN,0.000144
436,2024-08-28,AAPL-DVA,0.000039,0.000024,0.000013,1.370778e-05,0.000009,NaN,0.000011
437,2024-08-28,AAPL-FCEL,0.000110,0.000174,0.000095,9.532282e-05,0.000041,NaN,0.000062
438,2024-08-28,AAPL-NFLX,0.000057,0.000058,0.000057,4.955240e-05,0.000044,NaN,0.000047
439,2024-08-28,CAT-AAPL,0.000062,0.000063,0.000047,2.969448e-05,0.000021,NaN,0.000027
440,2024-08-28,CAT-CAT,0.000215,0.000238,0.000190,1.581256e-04,0.000140,NaN,0.000152
441,2024-08-28,CAT-DVA,0.000057,0.000008,0.000005,-6.023307e-07,0.000008,NaN,0.000006
442,2024-08-28,CAT-FCEL,0.000327,0.000357,0.000308,2.529056e-04,0.000233,NaN,0.000248
443,2024-08-28,CAT-NFLX,0.000096,0.000081,0.000053,3.803361e-05,0.000030,NaN,0.000036
444,2024-08-28,DVA-DVA,0.000182,0.000195,0.000159,1.391087e-04,0.000130,NaN,0.000137


In [15]:
df_dates_predict = df_dates_predict[['VALORES', 'output_predict']].rename(columns = {'output_predict': 'COV'})
df_dates_predict[['VALOR_1', 'VALOR_2']] = df_dates_predict['VALORES'].str.split('-', expand = True)
df_dates_predict = df_dates_predict.drop(columns = 'VALORES')[['VALOR_1', 'VALOR_2', 'COV']]
df_dates_predict = df_dates_predict.reset_index(drop = True)
df_dates_predict.to_csv(f'{cofre}Inputs_mkw/Covarianza_{output_level}.csv', sep = ';', decimal = ',', index = False)
df_dates_predict # Covarianza disponibe

NAME,VALOR_1,VALOR_2,COV
0,AAPL,AAPL,0.000144
1,AAPL,DVA,0.000011
2,AAPL,FCEL,0.000062
3,AAPL,NFLX,0.000047
4,CAT,AAPL,0.000027
5,CAT,CAT,0.000152
6,CAT,DVA,0.000006
7,CAT,FCEL,0.000248
8,CAT,NFLX,0.000036
9,DVA,DVA,0.000137
